<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 8 · Bond Futures and Duration Hedging

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch08_bond_futures_duration_hedging.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Bond futures adjust cash-bond duration without selling the portfolio, and the hedge is only as good as its DV01 arithmetic. In this lab you compute portfolio DV01 from market values and modified durations, translate a cheapest-to-deliver bond's DV01 into futures terms through the conversion factor, and size a short hedge that you then test under a parallel yield shock. You also rank a deliverable basket by net basis to see how the CTD bond is chosen, and you identify the rounding and CTD-switching effects that leave residual basis risk instead of a perfect offset.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The script computes a portfolio DV01 of 14,730 USD per basis point from three cash-bond positions, divides the CTD bond's DV01 of 72 by the 0.82 conversion factor for a futures DV01 of about 87.80 USD per basis point, and prints a signed hedge ratio that rounds to 168 short contracts. It then shocks yields by 25 basis points and reports the portfolio loss, the futures gain, and the small net profit left by whole-contract rounding.

In [ ]:
SCRIPT = ROOT / "code" / "ch08_bond_futures_hedging.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

1. Verify the portfolio DV01: positions of 12m at 4.2, 8m at 6.8, and 5m at 8.5 years sum to 14,730 USD per basis point once each product is divided by 10,000.
2. Confirm the futures DV01 of 72 / 0.82 = 87.80 USD per basis point and read the sign: a positive cash-bond DV01 requires a short futures position, about -167.76 contracts before rounding.
3. Check the scenario table: the portfolio loses 368,250 USD on the 25-basis-point rise while 168 short contracts gain 368,780.49 USD, leaving +530.49 USD of rounding residual.
4. Note that the futures DV01 inherits cheapest-to-deliver risk: if the CTD bond switches, the conversion-factor arithmetic and the hedge ratio change.
5. Treat the contract and position tables as illustrative teaching inputs, not quoted market prices.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch08_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Pick the CTD by net basis

The short futures position delivers whichever eligible bond is cheapest after conversion-factor adjustment. Net basis, defined as the cash price minus the futures price times the conversion factor, ranks the basket. This lab scores three illustrative deliverables and shows that the lowest cash price and the lowest net basis need not belong to the same bond.

In [ ]:
import pandas as pd

futures_price = 115.00  # quoted futures price (per 100 nominal)
basket = pd.DataFrame({
    "bond": ["A", "B", "C"],
    "cash_price": [99.75, 97.20, 101.10],  # clean prices per 100
    "cf": [0.82, 0.81, 0.865],  # conversion factors
})
basket["net_basis"] = basket.cash_price - futures_price * basket.cf
print(basket.round(4))
print("CTD bond:", basket.loc[basket.net_basis.idxmin(), "bond"])

### What to observe

Net bases are 5.4500 for A, 4.0500 for B, and 1.6250 for C, so bond C is cheapest to deliver even though it carries the highest cash price; its conversion factor delivers the most value per unit of futures price. At a futures price of 70.00 the ranking flips and B becomes the CTD, which is the switching risk behind every futures hedge.

## Exercises

1. **Rebuild the portfolio DV01.** Recompute the 14,730 USD per basis point total from the three positions and modified durations. *Hint:* each bond contributes market value times duration divided by 10,000.
2. **Stress the hedge.** Change bond_a's market value by ten percent and recompute the hedge ratio and the 25-basis-point scenario. *Hint:* only the portfolio DV01 changes; the futures leg follows through the same ratio.
3. **Explain the conversion factor.** State in two sentences why dividing the CTD DV01 by 0.82 moves cash-bond sensitivity into futures-price terms. *Hint:* the invoice price is the futures price times the conversion factor.
4. **Find the CTD switch point.** Solve for the futures price at which bonds B and C tie in the mini-lab basket. *Hint:* set the two net-basis expressions equal; the crossover sits near 71.
5. **Interpret the residual.** Explain why the hedged scenario ends at +530.49 USD rather than at zero. *Hint:* the unrounded hedge is about -167.76 contracts.
6. **Scale the shock.** Rerun the scenario at 100 basis points and judge whether the linear DV01 approximation still holds. *Hint:* compare against a duration-plus-convexity estimate.

For more detail, see Chapter 8 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">